# Testing and building confidence in models in Python

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/waelrash1/loopbricks/blob/main/notebooks/16-testing.ipynb)

No model can be proved true, but a model can be put through tests that it might fail, and every test it passes earns it a little more trust for a given purpose. This notebook runs four such tests on LoopBricks molecules: extreme conditions on [Reducing Backlog](https://waelrash1.github.io/loopbricks/#/m/reducingBacklog), [Level Protected by Flow](https://waelrash1.github.io/loopbricks/#/m/protFlow) and [Backlog Protected by Level](https://waelrash1.github.io/loopbricks/#/m/backlogLevel); a time step test on [Close Gap](https://waelrash1.github.io/loopbricks/#/m/closegap) and [Stock Management](https://waelrash1.github.io/loopbricks/#/m/stockmgmt); a loop knockout; and a sensitivity sweep on [Diffusion](https://waelrash1.github.io/loopbricks/#/m/diffusion).

It goes with the [Testing module](https://waelrash1.github.io/loopbricks/#/learn/testing) in the app.

In [ ]:
# Run this cell first. In Colab it installs the loopbricks package, which takes about ten seconds.
try:
    import loopbricks as lb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "loopbricks @ git+https://github.com/waelrash1/loopbricks.git@main#subdirectory=python"])
    import loopbricks as lb
import matplotlib.pyplot as plt
print("loopbricks", lb.__version__, "with molecules:", ", ".join(lb.molecules()))

## 1. Push it to the extremes

A model that behaves well near normal conditions can still be nonsense when a policy pushes it far away. The **extreme conditions test** asks: if an input takes an absurd value (no workers, a huge order, zero stock), does the model still do something that could happen? A warehouse with no stock cannot ship. A team with no work left cannot keep producing.

**Reducing Backlog** drains a stock of work at workers × productivity. Give it a big team and watch what happens when the work runs out. The `Work to do` stock is held at zero, so the trouble shows up in a second stock, `Overdrawn`, which counts production that was taken from nothing.

In [ ]:
rb = lb.molecule("reducingBacklog")
big = rb.run(workers=30, productivity=10)
big[["Work to do", "Overdrawn"]].plot(figsize=(8, 4), grid=True)
print(big.loc[[0, 3, 4, 10, 60], ["Work to do", "Producing", "Overdrawn"]])

Thirty people at 10 tasks a month clear 1,200 tasks in 4 months. After that the work is gone, yet `Producing` stays at 300 a month, and by month 60 the model has produced 16,800 tasks that never existed. The model fails the test. The reason is structural: the outflow does not depend on the stock it drains, so nothing slows it as the stock empties. Every outflow from a physical stock needs a first-order link back to that stock.

**Level Protected by Flow** has that link: draining is the smaller of what is wanted and what the stock can supply, `MIN(Desired_draining, On_hand / Fastest_draining_time)`. Give it an absurd demand and see.

In [ ]:
def extreme_test(model, **params):
    """Run a model with extreme settings and report the lowest value each stock reaches."""
    run = model.run(**params)
    for s in model.stocks:
        low = run[s["name"]].min()
        print(f"  {s['name']:<12} lowest {low:10.2f}{'   <- negative!' if low < -1e-9 else ''}")
    return run

pf = lb.molecule("protFlow")
print("Desired draining 1000 a month:")
run = extreme_test(pf, desired_draining=1000)
print(f"  draining at the end: {run['Draining'].iloc[-1]:.2f} a month, on hand {run['On hand'].iloc[-1]:.2f}")

Even when the outflow wants 1,000 a month, the stock never goes below zero. Draining falls as the stock falls, until it matches the replenishment of 3 a month and the stock rests at 3 × 2 = 6. The protected formulation passes.

**Try it.** **Backlog Protected by Level** ships orders only as far as inventory allows. Run two extremes: production set to zero, and orders set to zero. Does anything impossible happen?

<details><summary>Show a worked answer</summary>

The next cell runs both. With no production, inventory runs down towards zero but never below it, shipping falls with it, and unfilled orders pile up in the backlog (650 by month 60: 12 new orders arrive each month, but once the 100 units on hand have gone out nothing more can be shipped). With no orders, the backlog empties and inventory grows by the 12 a month still being produced, to 790 by month 60. Both are what a real warehouse would do, so the model passes. The second run also hints at a missing piece: no real firm keeps producing for a warehouse that nobody buys from. Extreme tests often show what lies outside a model's boundary.
</details>

In [ ]:
bl = lb.molecule("backlogLevel")
print("No production:")
no_prod = extreme_test(bl, production=0)
print(no_prod.loc[[0, 24, 60], ["Inventory", "Backlog", "Shipping rate"]].round(2))
print("No orders:")
no_orders = extreme_test(bl, order_rate=0)
print(no_orders.loc[[60], ["Inventory", "Backlog"]].round(2))

## 2. The time step must not matter

A simulator moves through time in steps of `dt`, holding every flow constant during a step. The results should belong to the model, not to that piece of arithmetic. The test is simple: run again with a smaller step. If the answer moves noticeably, the step was too big.

**Close Gap** with a 1-week adjustment time has an exact answer: after one week the stock has closed 1 − e⁻¹ of the gap, so it reaches 60 − 40 e⁻¹ ≈ 45.28.

In [ ]:
import numpy as np
cg = lb.molecule("closegap")
exact = 60 - 40 * np.exp(-1)
print(f"exact value after one week: {exact:.2f}")
for dt in (1, 0.5, 0.25, 0.125, 0.0625):
    value = cg.run(T=10, dt=dt, adjustment_time=1).loc[1.0, "Actual"]
    print(f"dt {dt:<6}: {value:6.2f}   error {value - exact:5.2f}")

A step as long as the adjustment time jumps straight to 60, an error of nearly 15. Each halving of the step roughly halves the error: 4.7, 2.1, 0.97, 0.47. That is the signature of Euler's method, whose error is proportional to `dt`. A common rule is to keep `dt` at a quarter to a tenth of the shortest time constant in the model, then check by halving it.

**Try it.** Apply the halving test to Stock Management with the supply line ignored (`supply_line_weight=0`), whose time constants are 4 and 6 weeks. Run with `dt` of 1, 0.5, 0.25 and 0.125 and compare the peak of inventory. Which steps would you trust?

<details><summary>Show a worked answer</summary>

The cell below prints the peaks: 114.1, 111.7, 110.7 and 110.3. Going from 1 to 0.5 moves the peak by more than 2 units; going from 0.25 to 0.125 moves it by less than half a unit, under 0.5% of the target, and the peak's timing barely moves (16.75 weeks both times). So 0.25 and smaller pass, which fits the rule of thumb: a quarter of the shortest time constant (4 weeks) is 1, a tenth is 0.4, and the app's 0.1 is safely below that. Any conclusion that depended on the difference between 114 and 110 would be a conclusion about arithmetic, not about inventories.
</details>

In [ ]:
sm = lb.molecule("stockmgmt")
for dt in (1, 0.5, 0.25, 0.125):
    inv = sm.run(T=100, dt=dt, supply_line_weight=0)["Inventory"]
    print(f"dt {dt:<5}: peak {inv.max():6.2f} at week {inv.idxmax()}")

## 3. Knock a loop out

Sometimes data cannot tell you whether a link in the model matters. Then switch it off and watch. If the behaviour becomes strange in a way that real systems do not show, the link was doing real work. If nothing changes, it may not be needed.

In **Stock Management** the supply line weight controls a whole balancing loop: the correction for goods already on order. Setting the weight to 0 knocks that loop out.

In [ ]:
def turning_points(series):
    """Times and values where a series turns, up or down."""
    v, t = series.values, series.index
    return [(float(t[i]), round(float(v[i]), 1)) for i in range(1, len(v) - 1)
            if v[i - 1] < v[i] >= v[i + 1] or v[i - 1] > v[i] <= v[i + 1]]

fig, ax = plt.subplots(figsize=(8, 4))
for w, label in ((1, "loop in"), (0, "loop knocked out")):
    inv = sm.run(T=100, supply_line_weight=w)["Inventory"]
    inv.plot(ax=ax, label=label)
    last_off = inv.index[(inv - 100).abs() > 1][-1]
    print(f"{label:<17}: turning points {turning_points(inv)[:3]}, within 1 of target after week {last_off}")
ax.axhline(100, ls=":", c="grey"); ax.set_ylabel("Inventory"); ax.legend(); ax.grid(alpha=0.3)

With the loop in place, inventory rises from 60 to 100 without a single turning point and is within 1 unit of the target by week 28. Knock it out and inventory overshoots to about 110 at week 17, dips to about 97 at week 33, and keeps ringing until week 40. Firms that track their orders do not behave like that, so the loop belongs in the model, and the knockout tells you what it does: it is the loop that stops the firm from ordering the same goods twice.

**Try it.** A good model should also explain its relatives (the **family member** test). Double the acquisition delay to 12 weeks, with the loop knocked out. Is the result the same kind of behaviour, and how does it scale?

<details><summary>Show a worked answer</summary>

The next cell runs it. With the loop knocked out, inventory overshoots to about 116 at week 23, dips to about 94 at week 45, and rings with peaks about 45 weeks apart instead of about 33. It is not within 1 unit of the target for good until week 77, against week 40 with the 6-week delay. Same structure, same kind of damped cycle, but bigger and slower, which is what you would expect from a longer delay. With the loop restored, the 12-week case rises to 100 with no overshoot, though it takes until week 49 to get within 1 unit (week 28 with the 6-week delay).
</details>

In [ ]:
for w in (0, 1):
    inv = sm.run(T=150, acquisition_delay=12, supply_line_weight=w)["Inventory"]
    last_off = inv.index[(inv - 100).abs() > 1][-1]
    print(f"weight {w}: turning points {turning_points(inv)[:4]}, within 1 of target after week {last_off}")

## 4. How much do the assumptions matter?

Many numbers in a model are uncertain. A **sensitivity analysis** varies them over a believable range and asks whether the conclusions change. In **Diffusion**, two numbers drive adoption of a new product among 1,000 people: `Innovation p`, the percentage who adopt each week on their own, and `Imitation q`, the strength of word of mouth. We measure one outcome: the week when half the people have adopted.

In [ ]:
import pandas as pd
diff = lb.molecule("diffusion")

def half_time(**params):
    adopters = diff.run(T=100, **params)["Adopters"]
    return float(adopters.index[adopters >= 500][0])

table = pd.DataFrame({f"q = {q}": [half_time(innovation_p=p, imitation_q=q) for p in (0.5, 1, 2)] for q in (10, 18, 27)},
                     index=["p = 0.5", "p = 1", "p = 2"])
print("Week when half have adopted:")
print(table)

Over these ranges the half-way week runs from about 30 (slow on both counts) to under 10 (fast on both). That is a big spread for the timing, so a plan that depends on *when* the market fills needs better estimates of p and q. But in every case the curve is the same S shape and everyone adopts in the end: the qualitative conclusion does not depend on the numbers. Sensitivity analysis is about which conclusions are robust, not only about how much the numbers move.

**Try it.** Instead of a grid, draw 200 random pairs, p between 0.5 and 2 and q between 10 and 27 (use `np.random.default_rng(1)` so you get the same draws). What range covers the middle 90% of the half-way weeks? Which of the two numbers matters more?

<details><summary>Show a worked answer</summary>

The cell below does it. The 5th, 50th and 95th percentiles are about 11, 15 and 23 weeks. The correlation of the half-way week with q is about −0.76 and with p about −0.61, so over these ranges q (word of mouth) matters a little more. Both correlations are negative because more of either speeds adoption. A real study would choose the ranges from evidence, because a sensitivity result is only as good as the ranges you feed it.
</details>

In [ ]:
rng = np.random.default_rng(1)
ps, qs = rng.uniform(0.5, 2, 200), rng.uniform(10, 27, 200)
weeks = np.array([half_time(innovation_p=p, imitation_q=q) for p, q in zip(ps, qs)])
print("5th, 50th, 95th percentiles:", np.percentile(weeks, [5, 50, 95]).round(1))
print(f"correlation with p: {np.corrcoef(ps, weeks)[0, 1]:.2f}, with q: {np.corrcoef(qs, weeks)[0, 1]:.2f}")
plt.hist(weeks, bins=20); plt.xlabel("week when half have adopted"); plt.ylabel("runs"); plt.grid(alpha=0.3)

## Where next

- Back in the app: the [Testing module](https://waelrash1.github.io/loopbricks/#/learn/testing) covers all twelve kinds of test, including those for boundary, units and fit to data.
- The [Validation](https://waelrash1.github.io/loopbricks/#/methods/validation), [Calibration lab](https://waelrash1.github.io/loopbricks/#/methods/calibration) and [Sensitivity lab](https://waelrash1.github.io/loopbricks/#/methods/sensitivity) pages go deeper.
- Why the time step matters, and how to choose it, is the subject of [Simulation mechanics](https://waelrash1.github.io/loopbricks/#/learn/mechanics). Fitting parameters to data is one of the tools in [Where system dynamics is going](https://waelrash1.github.io/loopbricks/#/learn/frontier).
- Testing starts with framing the problem: see [The modelling process](https://waelrash1.github.io/loopbricks/#/learn/process).